# Reproducing Hiraoka & Jo (2025) on DANDI:000044: the representative neuron

Paper: Takayuki Hiraoka and Hang-Hyun Jo, "Hierarchical organization of bursty trains in event
sequences", doi:10.1063/5.0296506. The claims file (`claims/hiraoka-2025.yaml`) was written from
`paper/10.1063-5.0296506.txt`, whose body is the arXiv preprint (arXiv:2508.18281v1, dated
September 22, 2025); the paper has no supplement and links no code.

Dataset: DANDI:000044 (Grosmark and Buzsaki 2016), version 0.210812.1516, DOI
10.48324/dandi.000044/0.210812.1516. Why this version: the paper cites it (ref. 39,
https://dandiarchive.org/dandiset/000044/0.210812.1516), and the claims file gives it. Later
versions exist (latest 0.261106.0236); they re-export the files, renumber the units and rename
the cell types, which matters for finding "unit 9" (shown in the version check below). The spike
trains themselves are the same in the representative session.

Files are streamed from the archive with `remfile` and `h5py`; nothing is downloaded to disk.
The helper `bursttree.py` holds the loading, the burst and merging-number measures, the fits and
the paper's two algorithms, and says where each parameter comes from.

Notebooks of this entry: this one (data check and the representative neuron),
`other_neurons.ipynb` (fig2-other-neurons-similar), `other_neurons_cap3.ipynb` (its variant with the
exponent capped at 3) and `model.ipynb` (Section IV, Figure 4).

| Claim | Tested where | Status |
|---|---|---|
| fig2-neuron-representative-sequence | this notebook | tested |
| fig2-iet-broad | this notebook | tested |
| fig2-neuron-burst-size-heavy-tail | this notebook | tested |
| fig2-neuron-beta-stable | this notebook | tested |
| fig2-neuron-beta-decrease-follows-gamma | this notebook | tested |
| fig2-neuron-merging-heavy-tail | this notebook | tested |
| fig2-neuron-merging-collapse | this notebook | tested |
| fig2-exponents-above-two | this notebook | tested |
| fig2-other-neurons-similar | other_neurons.ipynb, other_neurons_cap3.ipynb | tested |
| fig2-wikipedia-beta | none | blocked (`too_large`): the paper's preprocessed edits are not public; the October 2015 dump survives on the Internet Archive (item enwiki-20151002, stub-meta-history 47.4 GB compressed) |
| fig2-earthquake-beta | none | blocked (`other`): the SCSN catalog is public but continuously revised and has no versioned release to pin (author, step 1.4) |
| fig4-model-heavy-tails | model.ipynb | tested (data generated by Algorithm 2) |
| fig4-model-iet-shapes | model.ipynb | tested (data generated by Algorithm 2) |
| model-decremental-equivalent | model.ipynb | tested (data generated by Algorithms 1 and 2) |

Open details O1 to O4 are those of the claims file's opening comment. Every result is shown
under each reading they name; the outcomes say which readings they rest on.

In [ ]:
import time, resource, zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import powerlaw
import bursttree as bt

T0 = time.time()
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 200)
PAPER = "hiraoka-2025"
print("dandiset", bt.DANDISET, "version", bt.DANDI_VERSION)
print("powerlaw", powerlaw.__version__, "; adopted readings (unbounded exponent):", bt.ADOPTED_READINGS, "; capped variant:", bt.CAPPED_READINGS)

## Data check

Every session of the pinned version is opened and its units table read (ids, every column,
spike times). Spike times are stored in seconds; the analysis uses milliseconds, as the paper
does. The checks: counts of units by session and cell type, that spike times are sorted, that
they fall inside the session's epochs, the epoch durations against the paper's description
(two four-hour rest periods around a 45-minute maze session), and whether any spike train is
deposited twice (identical spike count and times, within or across sessions).

In [ ]:
assets = bt.list_assets()
rows, units, epochs = [], {}, {}
for a in assets:
    ses = a["path"].split("ses-")[1].split("_")[0]
    f = bt.open_nwb(a["url"])
    U = bt.load_units(f); E = bt.load_epochs(f)
    units[ses], epochs[ses] = U, E
    allsp = np.concatenate(U["spikes"])
    rows.append(dict(session=ses, subject=a["path"].split("/")[0], size_GB=round(a["size"] / 1e9, 2),
                     units=len(U["id"]), excitatory=int((U["cell_type"] == "excitatory").sum()),
                     inhibitory=int((U["cell_type"] == "inhibitory").sum()),
                     columns=",".join(k for k in U if k != "spikes"),
                     spikes=len(allsp), t_first_s=round(allsp.min(), 2), t_last_s=round(allsp.max(), 2),
                     all_sorted=all(np.all(np.diff(s) >= 0) for s in U["spikes"]),
                     min_iet_ms=round(min(np.diff(s).min() for s in U["spikes"] if len(s) > 1) * 1e3, 3),
                     epochs=" ".join(f"{l}:{(b - a_) / 3600:.2f}h" for l, a_, b in zip(E["label"], E["start"], E["stop"])),
                     session_end_s=round(E["stop"].max(), 1)))
survey = pd.DataFrame(rows)
print(survey.to_string())
print("units", survey.units.sum(), "excitatory", survey.excitatory.sum(), "inhibitory", survey.inhibitory.sum())
print("locations:", {s: dict(zip(*np.unique(u["location"], return_counts=True))) for s, u in units.items()})
# spikes inside the epochs
for s, U in units.items():
    allsp = np.concatenate(U["spikes"]); E = epochs[s]
    inside = np.zeros(len(allsp), bool)
    for a_, b in zip(E["start"], E["stop"]):
        inside |= (allsp >= a_) & (allsp <= b)
    print(f"{s}: fraction of spikes inside an epoch {inside.mean():.6f}; spikes after the last epoch ends ({E['stop'].max():.1f} s): {(allsp > E['stop'].max()).sum()}, up to {allsp.max():.1f} s")
# spike trains not sorted in time: where they break
for s, U in units.items():
    for i, sp in enumerate(U["spikes"]):
        br = np.flatnonzero(np.diff(sp) < 0)
        if len(br):
            print(f"{s}: unit id {U['id'][i]} ({U['cell_type'][i]}), {len(sp)} spikes, {len(br)} break(s) after index {br.tolist()}; "
                  f"first run {sp[0]:.2f} to {sp[br[0]]:.2f} s ({br[0] + 1} spikes), second run {sp[br[0] + 1]:.2f} to {sp[-1]:.2f} s")
# duplicates: identical spike trains
keys = {}
for s, U in units.items():
    for i, sp in enumerate(U["spikes"]):
        keys.setdefault((len(sp), round(float(sp[0]), 6), round(float(sp[-1]), 6)), []).append((s, int(U["id"][i])))
dups = {k: v for k, v in keys.items() if len(v) > 1}
print("spike trains sharing count, first and last spike:", len(dups), dups if dups else "")
same_counts = pd.Series({s: tuple(np.sort([len(x) for x in U["spikes"]])) for s, U in units.items()})
print("sessions with identical sorted spike-count lists:", same_counts.duplicated(keep=False).sum())

### Clock of the spikes and the behaviour

The analysis uses spike times only; the epochs are used for one variant. To check that spikes
and the maze-epoch position share a clock, the summed rate of the excitatory units of the
representative session in 1 s bins is correlated with running speed, with the position shifted
in time. A shared clock gives a peak near zero shift. The position series stores
`starting_time` with an attribute `rate` (read below), which is checked against the epoch.

In [ ]:
f = bt.open_nwb([a["url"] for a in assets if "Achilles-10252013" in a["path"]][0])
ps = f["processing/behavior/1.6mLinearMazePosition/1.6mLinearMazeSpatialSeries"]
pos = ps["data"][:]; t_start = float(ps["starting_time"][()]); rate_attr = float(ps["starting_time"].attrs["rate"])
E = epochs["Achilles-10252013"]
maze = dict(zip(E["label"], zip(E["start"], E["stop"])))["MazeEpoch"]
print(f"position samples {len(pos)}, starting_time {t_start} s, 'rate' attribute {rate_attr:.6f}")
print(f"span if 'rate' is in Hz: {len(pos) / rate_attr:.0f} s; span if it is the sampling interval: {len(pos) * rate_attr:.1f} s; maze epoch {maze[1] - maze[0]:.1f} s")
print(f"position range x {np.nanmin(pos[:,0]):.2f} to {np.nanmax(pos[:,0]):.2f} m, y {np.nanmin(pos[:,1]):.2f} to {np.nanmax(pos[:,1]):.2f} m; NaN fraction {np.isnan(pos[:,0]).mean():.3f}")
tp = t_start + rate_attr * np.arange(len(pos))  # the interval reading, which fits the epoch
edges = np.arange(maze[0], maze[1], 1.0)
idx = np.clip(np.searchsorted(tp, edges), 0, len(pos) - 1)
p1 = pos[idx]
speed = np.r_[np.hypot(*np.diff(p1, axis=0).T), np.nan]
U = units["Achilles-10252013"]
pop = sum(bt.spike_counts(s, np.r_[edges, edges[-1] + 1]) for s, c in zip(U["spikes"], U["cell_type"]) if c == "excitatory").astype(float)
shifts = np.arange(-120, 121, 5)
cc = []
for sh in shifts:
    a_ = speed[max(0, sh): len(speed) + min(0, sh)]; b = pop[max(0, -sh): len(pop) + min(0, -sh)]
    ok = np.isfinite(a_) & np.isfinite(b)
    cc.append(np.corrcoef(a_[ok], b[ok])[0, 1])
cc = np.array(cc)
print("correlation of excitatory population rate with speed, by shift of the position (s):")
print(pd.Series(cc.round(3), index=shifts).to_string())
print(f"peak at shift {shifts[np.nanargmax(cc)]} s, r = {np.nanmax(cc):.3f}; median |r| at shifts beyond 60 s {np.nanmedian(np.abs(cc[np.abs(shifts) > 60])):.3f}")

### Version check

The paper cites version 0.210812.1516 (ref. 39), pinned here. The latest published version,
0.261106.0236, renumbers the units and relabels the cell types; its units table for the
representative session is read to show the difference for the unit the claims use.

In [ ]:
latest = [a for a in bt.list_assets(bt.LATEST_VERSION) if "Achilles-10252013" in a["path"]][0]
fl = bt.open_nwb(latest["url"]); UL = bt.load_units(fl)
U = units["Achilles-10252013"]
print("cited version columns:", [k for k in U if k != "spikes"])
print("latest version columns:", [k for k in UL if k != "spikes"])
cmp = pd.DataFrame({"cited_id": U["id"][:12], "cited_type": U["cell_type"][:12], "cited_n": [len(s) for s in U["spikes"][:12]],
                    "latest_id": UL["id"][:12], "latest_type": UL["cell_type"][:12],
                    "latest_cluster_id": UL.get("cluster_id", np.full(len(UL["id"]), -1))[:12], "latest_n": [len(s) for s in UL["spikes"][:12]]})
print(cmp.to_string())
same = len(U["spikes"]) == len(UL["spikes"]) and all(len(a_) == len(b) and np.allclose(a_, b) for a_, b in zip(U["spikes"], UL["spikes"]))
print("same spike trains in the same order in both versions:", same)

## fig2-neuron-representative-sequence

> The representative neuronal sequence (unit 9 of session Achilles-10252013) has 228 356
> spikes, with IETs from 1.05 ms to 1.82 x 10^4 ms.

Criterion: the unit is found as in O1 (readings (a) units id 9, (b) row 9 counted from 0,
(c) the ninth unit counted from 1, (d) a cluster number 9 in the unit metadata, here the
`shank_id` column, described as "0-indexed id of cluster identified from the shank"), with all
spikes of the session (O2). Reproduced when under one reading all three values are within
tolerance (n 228 356 +/- 22 836, tau_min 1.05 +/- 0.105 ms, tau_max 18 200 +/- 1 820 ms).

In [ ]:
U = units["Achilles-10252013"]
def stats(i):
    s = U["spikes"][i] * 1e3; d = np.diff(s)
    return dict(row=i, id=int(U["id"][i]), cell_type=U["cell_type"][i], shank_id=int(U["shank_id"][i]), n=len(s),
                tau_min_ms=round(d.min(), 3), tau_max_ms=round(d.max(), 1),
                match=bool(abs(len(s) - 228356) <= 22835.6 and abs(d.min() - 1.05) <= 0.105 and abs(d.max() - 18200) <= 1820))
o1 = []
o1 += [dict(reading="(a) id 9", **stats(i)) for i in np.flatnonzero(U["id"] == 9)]
o1 += [dict(reading="(b) row 9 from 0", **stats(9))]
o1 += [dict(reading="(c) ninth from 1", **stats(8))]
o1 += [dict(reading="(d) shank_id 9", **stats(i)) for i in np.flatnonzero(U["shank_id"] == 9)]
o1 = pd.DataFrame(o1); print(o1.to_string())
REP = 9
print("units in the session matching all three values:", [int(U["id"][i]) for i in range(len(U["id"])) if stats(i)["match"]])
# the same row in the latest version
print("latest version, row 9:", dict(id=int(UL["id"][9]), cell_type=UL["cell_type"][9], n=len(UL["spikes"][9])),
      "; ids equal to 9 in latest:", int((UL["id"] == 9).sum()))
spk = U["spikes"][REP] * 1e3
tau = bt.iets(spk)
print(f"representative: n = {len(spk)}, tau_min = {tau.min():.3f} ms, tau_max = {tau.max():.1f} ms, cell_type {U['cell_type'][REP]}, location {U['location'][REP]}")

**Result: reproduced.** Under reading (b) of O1, row 9 of the units table counted from 0 (unit id
18), the unit has 228 356 spikes, tau_min 1.050 ms and tau_max 18 229.7 ms, all within tolerance,
and it is the only unit of the session that matches. Readings (a), (c) and (d) do not match. The
files label this unit `inhibitory` (`interneuron` in the latest version), while the paper
describes its neuronal data as pyramidal cells. In the latest version the same row has id 119 and
no unit has id 9.

## fig2-iet-broad

> The IETs of each process, the representative neuron included, extend across several orders
> of magnitude.

Criterion: for the representative unit, reproduced when tau_max / tau_min >= 10^3.

In [ ]:
ratio = tau.max() / tau.min()
print(f"tau_max / tau_min = {ratio:.0f} ({np.log10(ratio):.2f} decades); paper 1.82e4 / 1.05 = {18200 / 1.05:.0f} ({np.log10(18200 / 1.05):.2f} decades)")
print("IET percentiles (ms):", dict(zip([0, 1, 50, 99, 100], np.percentile(tau, [0, 1, 50, 99, 100]).round(2))))
h, e = np.histogram(tau, np.logspace(0, 4.5, 46), density=True)
fig, ax = plt.subplots(figsize=(4, 3)); ax.loglog(np.sqrt(e[1:] * e[:-1]), h, "o-", ms=3)
ax.set_xlabel("tau (ms)"); ax.set_ylabel("P(tau)"); ax.set_title("IETs, representative unit")
print(f"IETs outside the plotted 1 to {10**4.5:.0f} ms: {(tau < 1).sum() + (tau > 10**4.5).sum()}")

**Result: reproduced.** tau_max / tau_min = 17 362, 4.24 decades, above the three decades of the
criterion and the same as the paper's 1.82 x 10^4 / 1.05 (4.24 decades).

## Bursts and merging numbers of the representative unit

Relevant timescales, grids, bursts and fits as in the Methods and in O3 and O4 of the claims
file (the helper says where each detail comes from). Both grids are run under the four fit
readings, each with the exponent unbounded (adopted) and capped at 3 (suffix `_cap3`, the default of
the locked powerlaw 2.0.0; helper, `READINGS`). The pairs of Figure 2 (delta' = 55, 100, 180, 330, 590 ms; delta = 3.3, 5.9 and 11
delta') are added as fixed pairs for fig2-neuron-merging-collapse.

In [ ]:
FIG2_PAIRS = [(dp, r * dp, r) for dp in (55, 100, 180, 330, 590) for r in bt.NEURON_RATIOS]
t1 = time.time()
RES = {g: bt.analyse_sequence(spk, g, extra_pairs=FIG2_PAIRS if g == "b" else ()) for g in bt.GRIDS}
print(f"fits done in {time.time() - t1:.0f} s")
lo, hi = RES["a"]["lo"], RES["a"]["hi"]
print(f"relevant timescales: {lo:.2f} ms (smallest delta with a burst of >= 100 events) to {hi:.2f} ms (the 99th largest IET; delta must stay below it for >= 100 bursts)")
print(f"paper (Figure 2 labels): about 55 to 6300 ms")
print("bursts (m_delta) at delta = 6282.75 ms (the upper end, excluded) and at the 6300 ms label of Figure 2:",
      int((tau > hi).sum()) + 1, "and", int((tau > 6300).sum()) + 1, "(at least 100 needed)")
for g in bt.GRIDS:
    print(f"grid ({g}): {len(RES[g]['grid'])} points, step log10 {np.log10(RES[g]['grid'][1] / RES[g]['grid'][0]):.4f}:", RES[g]["grid"].round(1))
B = pd.concat([pd.DataFrame(RES[g]["bursts"]).assign(grid=g) for g in bt.GRIDS], ignore_index=True)
P = pd.concat([pd.DataFrame(RES[g]["pairs"]).assign(grid=g) for g in bt.GRIDS], ignore_index=True)
print("fits with a failed test (R or p not finite):", int((~np.isfinite(B.R)).sum() + (~np.isfinite(P.R)).sum()), "of", len(B) + len(P))
# the exponent range (helper, READINGS): capped fits at the bound of powerlaw 2.0.0, and capped fits that differ from the unbounded ones
cap = []
for nm, D in (("bursts", B), ("pairs", P)):
    key_ = ["grid", "delta"] + (["dprime"] if nm == "pairs" else [])
    for r in bt.ADOPTED_READINGS:
        u = D[D.reading == r].set_index(key_).alpha; c3 = D[D.reading == r + "_cap3"].set_index(key_)
        cap.append(dict(fits=nm, reading=r, n=len(u), unbounded_above_3=int((u > 3).sum()), unbounded_max=round(u.max(), 3),
                        capped_at_bound=int(c3.at_cap.sum()), capped_differs_by_0_01=int(((c3.alpha - u).abs() > 0.01).sum())))
print("\nexponent range, unbounded against capped at 3 (both grids, Figure 2 pairs included):"); print(pd.DataFrame(cap).to_string())
print("unbounded fits out of the range [1, inf):", int((~B[B.reading.isin(bt.ADOPTED_READINGS)].in_range).sum() + (~P[P.reading.isin(bt.ADOPTED_READINGS)].in_range).sum()))

**The range of the exponent.** The locked powerlaw 2.0.0 caps the power-law exponent at 3 by
default, and its choice of xmin by the KS distance discards every candidate whose fit reaches the
cap. The table above compares the capped fits with the unbounded ones adopted here: under the
discrete KS fit 4 of 156 burst-size fits and 14 of 106 merging-number fits exceed 3 when
unbounded (largest 3.358 and 3.795), and the capped fits differ from them in exactly those cases,
though none of them sits at the bound, because the cap moves xmin. Under the continuous KS fit
7 and 33 fits exceed 3. No unbounded fit is out of its range.

## fig2-neuron-burst-size-heavy-tail

> For the representative neuron, the burst size distributions are heavy-tailed across different
> timescales.

Criterion: at each delta of the grid (O4) over the relevant range, fit and test as in O3.
Reproduced when the power law is supported (R > 0, p < 0.05) at more than half of the relevant
timescales and at every one below 3000 ms; partially reproduced when at more than half but not
at every one below 3000 ms. As a reference, the same test is run on the IETs shuffled in time
(a renewal process, whose burst sizes decay exponentially; Introduction), under the claim's
seed and four more.

In [ ]:
Bg = B[B.on_grid]
summ = []
for (g, r), d in Bg.groupby(["grid", "reading"]):
    below = d[d.delta < 3000]
    summ.append(dict(grid=g, reading=r, n_delta=len(d), supported=int(d.supported.sum()), frac=round(d.supported.mean(), 3),
                     n_below3000=len(below), supported_below3000=int(below.supported.sum()),
                     more_than_half=bool(d.supported.sum() > len(d) / 2), all_below3000=bool(below.supported.all())))
summ = pd.DataFrame(summ); print(summ.to_string())
print("\nper delta, grid (a):")
print(Bg[Bg.grid == "a"].pivot_table(index="delta", columns="reading", values=["alpha", "R", "p"]).round(4).to_string())
print("\nper delta, grid (b), supported (1) or not (0):")
print(Bg[Bg.grid == "b"].pivot_table(index="delta", columns="reading", values="supported").astype(int).to_string())
print("\nunsupported, grid (b): ", Bg[(Bg.grid == "b") & ~Bg.supported][["reading", "delta", "alpha", "R", "p", "m"]].round(4).to_string())
# reference: shuffled IETs
seed = zlib.crc32(f"{PAPER}/fig2-neuron-burst-size-heavy-tail".encode())
ref = []
for off in (0, 1000, 2000, 3000, 4000):
    rng = np.random.default_rng(seed + off)
    ts = np.r_[0, np.cumsum(rng.permutation(tau))]
    tt = bt.iets(ts); g_ = bt.grid(*bt.relevant_range(tt), "a")
    sup = [bt.fit(bt.burst_sizes(tt, d), "ks_discrete")["supported"] for d in g_]
    ref.append(dict(seed=seed + off, n_delta=len(g_), supported=int(sum(sup))))
print("\nshuffled IETs (renewal reference), grid (a), ks_discrete:"); print(pd.DataFrame(ref).to_string())

**Result: reproduced.** With xmin chosen by the KS distance, discrete or continuous, the power
law is supported at every relevant timescale on both grids (8 of 8 on grid (a), 40 of 40 on grid
(b)), so at every one below 3000 ms, with the exponent unbounded or capped. The claim holds under
one reading of O3, as the claims file's rule asks. With xmin = 1 the discrete fit is supported at
5 of 8 and 29 of 40 timescales, not all below 3000 ms; the continuous fit at 7 of 8 (all 7 below
3000 ms) on grid (a) and 32 of 40 (32 of 34 below 3000 ms) on grid (b). The shuffled-IET
reference has no supported timescale under any of the five seeds (0 of 5 or 6).

## fig2-neuron-beta-stable

> For neuronal spikes the fitted burst-size exponent stays near 2.35 for several decades of
> timescales, up to about 3000 ms.

Criterion: beta at each delta of the grid from the lower end of the relevant range to 3000 ms,
fit of O3. Readings: (a) every estimate within 2.35 +/- 0.235; (b) the median within that band
and the range under 0.47. Reproduced when (a) or (b) holds under one reading of O3.

In [ ]:
st = []
for (g, r), d in Bg[Bg.delta < 3000].groupby(["grid", "reading"]):
    a_ = d.alpha.values
    st.append(dict(grid=g, reading=r, n=len(a_), median=round(np.median(a_), 3), min=round(a_.min(), 3), max=round(a_.max(), 3),
                   range=round(np.ptp(a_), 3), max_sigma=round(d.sigma.max(), 3), n_in_band=int((np.abs(a_ - 2.35) <= 0.235).sum()),
                   first_delta_all_in_band_after=round(min([dd for dd in d.delta if np.all(np.abs(d.alpha[d.delta >= dd] - 2.35) <= 0.235)], default=np.nan), 1),
                   all_in_band=bool(np.all(np.abs(a_ - 2.35) <= 0.235)),
                   median_in_band_range_ok=bool(abs(np.median(a_) - 2.35) <= 0.235 and np.ptp(a_) < 0.47)))
st = pd.DataFrame(st); print(st.to_string())
print("\nbeta per delta below 3000 ms (grid a), with standard errors:")
print(Bg[(Bg.grid == "a") & (Bg.delta < 3000)].pivot_table(index="delta", columns="reading", values=["alpha", "sigma", "xmin"]).round(3).to_string())
fig, ax = plt.subplots(figsize=(5, 3))
for r, d in Bg[Bg.grid == "b"].groupby("reading"):
    ax.semilogx(d.delta, d.alpha, "o-", ms=3, label=r)
ax.axhspan(2.35 - 0.235, 2.35 + 0.235, color="0.9"); ax.axvline(3000, ls=":", c="k")
ax.set_xlabel("delta (ms)"); ax.set_ylabel("beta"); ax.legend(fontsize=7)
print("plotted beta range (grid b):", Bg[Bg.grid == "b"].alpha.min().round(3), "to", Bg[Bg.grid == "b"].alpha.max().round(3))

**Result: not reproduced.** Neither reading of the criterion holds under any fit or grid. Under
the discrete KS fit on grid (b) the median beta below 3000 ms is 2.354, inside the band, and 27
of 34 estimates lie within 2.35 +/- 0.235, every one from 126.5 ms to 3000 ms (1.4 decades). At
the lower end beta is higher (3.349 at 55 ms), so the range is 1.079 against the 0.47 of reading
(b). With the exponent unbounded the ranges are 0.679 to 1.360 over the fits and grids (medians
2.316 to 2.433 under the KS fits, 1.521 to 1.750 with xmin = 1). Capped at 3, the KS ranges are
0.552 to 0.705, which still fail. The standard errors are at most 0.088.

## fig2-neuron-merging-heavy-tail

> For the representative neuron, the merging-number distributions are heavy-tailed for various
> pairs of timescales, and their fitted exponents gamma stay fairly constant across a wide range
> of timescales.

Criterion: pairs delta = 3.3, 5.9 and 11 delta', delta' on the grid (O4), both in the relevant
range; fit and test as in O3. Reproduced when the power law is supported for more than half of
the pairs and the gamma of those pairs has an interquartile range below 0.5; partially
reproduced when only the first holds.

In [ ]:
Pg = P[~P.extra]
mh = []
for (g, r), d in Pg.groupby(["grid", "reading"]):
    s = d[d.supported].alpha
    iqr = float(np.subtract(*np.percentile(s, [75, 25]))) if len(s) else np.nan
    mh.append(dict(grid=g, reading=r, n_pairs=len(d), supported=int(d.supported.sum()), frac=round(d.supported.mean(), 3),
                   gamma_median=round(s.median(), 3) if len(s) else np.nan, gamma_iqr=round(iqr, 3),
                   gamma_min=round(s.min(), 3) if len(s) else np.nan, gamma_max=round(s.max(), 3) if len(s) else np.nan,
                   more_than_half=bool(d.supported.sum() > len(d) / 2), iqr_below_05=bool(iqr < 0.5)))
mh = pd.DataFrame(mh); print(mh.to_string())
print("\ngamma per pair, grid (a):")
print(Pg[Pg.grid == "a"].pivot_table(index=["ratio", "dprime"], columns="reading", values="alpha").round(3).to_string())
print("\nsupported per pair, grid (a):")
print(Pg[Pg.grid == "a"].pivot_table(index=["ratio", "dprime"], columns="reading", values="supported").astype(int).to_string())

**Result: reproduced, under the discrete KS fit.** With the exponent unbounded, the discrete KS
fit supports the power law for 13 of 15 pairs (grid (a)) and 70 of 76 (grid (b)), and the gamma of
the supported pairs has an interquartile range of 0.378 and 0.415 (median 2.618 and 2.644). The
continuous KS fit supports 15 of 15 and 69 of 76, but its interquartile range is 0.642 and 0.546;
capped at 3 it was 0.408 and 0.426, so the cap decides that reading but not the discrete one
(0.383 and 0.401 capped). With xmin = 1 the discrete fit is supported for 6 of 15 and 19 of 76
pairs, and the continuous fit has an interquartile range of 0.673 and 0.597.

## fig2-neuron-merging-collapse

> Rescaled merging-number distributions for different pairs of delta and delta' collapse onto
> each other, so the burst tree is self-similar across levels.

Criterion, the pairs of Figure 2 (delta' = 55, 100, 180, 330, 590 ms, each with delta = 3.3,
5.9 and 11 delta'): (a) the complementary cumulative distributions of k / <k> differ by no more
than a factor of 2 at every point where all are above 10^-3; (b) the gamma estimates lie within
+/- 0.5 of their median (under a reading of O3). Reproduced when either holds. The points are
200 values of k / <k> spaced logarithmically from 0.01 to 1000 (a standard choice).

In [ ]:
xs = np.logspace(-2, 3, 200)
curves = {}
for dp, d, r in FIG2_PAIRS:
    k = bt.merging_numbers(tau, dp, d)
    curves[(dp, r)] = bt.ccdf_at(k / k.mean(), xs)
C = np.array(list(curves.values()))
ok = np.all(C > 1e-3, axis=0)
fac = C[:, ok].max(axis=0) / C[:, ok].min(axis=0)
print(f"pairs: {len(curves)}; points where all CCDFs exceed 1e-3: {ok.sum()} (k/<k> from {xs[ok].min():.3f} to {xs[ok].max():.2f})")
print(f"largest ratio between the highest and lowest CCDF at those points: {fac.max():.2f} at k/<k> = {xs[ok][fac.argmax()]:.2f}; median ratio {np.median(fac):.2f}")
print(f"(a) holds: {bool(fac.max() <= 2)}")
# excluding the smallest k/<k> where CCDF = 1 for all, how do the curves differ in the tail only (k/<k> >= 1)?
tail = ok & (xs >= 1)
print(f"in the tail alone (k/<k> >= 1): max ratio {(C[:, tail].max(0) / C[:, tail].min(0)).max():.2f}")
Pf = P[P.extra]
cg = []
for r_, d in Pf.groupby("reading"):
    med = d.alpha.median()
    cg.append(dict(reading=r_, median=round(med, 3), max_dev=round((d.alpha - med).abs().max(), 3),
                   within_05=bool(((d.alpha - med).abs() <= 0.5).all()), supported=int(d.supported.sum()), m_min=int(d.m.min())))
print(pd.DataFrame(cg).to_string())
print(Pf.pivot_table(index=["dprime", "ratio"], columns="reading", values="alpha").round(3).to_string())
print("pairs of Figure 2 with delta beyond the relevant range:", [(dp, round(d, 0)) for dp, d, r in FIG2_PAIRS if d >= hi])
fig, ax = plt.subplots(figsize=(4, 3))
for (dp, r), c in curves.items():
    ax.loglog(xs, np.where(c > 0, c, np.nan), lw=0.8)
ax.set_xlabel("k / <k>"); ax.set_ylabel("P(K >= k)"); ax.set_ylim(1e-5, 1.5)

**Result: not reproduced with the exponent unbounded (an assumption); reproduced only when it is
capped at 3.** Reading (b): with the exponent unbounded, the gamma of the 15 pairs of Figure 2
lie up to 0.538 from their median under the discrete KS fit (median 2.610), 1.742 under the
continuous KS fit, 0.521 under the discrete fit from xmin = 1 and 1.249 under the continuous one,
all outside the band of 0.5. The deviation of 0.538 is the pair delta' 180 ms, delta 594 ms
(gamma 3.148), which the cap holds at 2.843, giving the 0.490 of the capped discrete KS fit, the
only fit within the band. Reading (a) fails: the CCDFs of k / <k> differ by up to a factor of
12.54 (median 1.63) where all exceed 10^-3. One pair of Figure 2 (delta' 590 ms, delta 6490 ms)
lies above the relevant range, with 89 bursts. The margin is small (0.538 and 0.490 against
0.5), and O3 (a) names the package's default, which in the locked powerlaw 2.0.0 is the capped
fit: read that way the claim would be reproduced. The unbounded exponent is kept because the method
the paper cites has no cap and the cap clips the very estimate that decides the band; whether the
paper's fits were capped is asked of the authors (record, findings).

## fig2-exponents-above-two

> In most cases both gamma and the burst-size exponent at the smaller timescale exceed 2, so the
> scaling relation reduces to beta at delta equal to the smaller of gamma and beta at delta',
> which agrees with the observations.

Criterion, for the pairs of fig2-neuron-merging-heavy-tail with the fit of O3: (1) in more than
half of the pairs both gamma and beta(delta') exceed 2; (2) in more than half, beta(delta) is
within 10% of min(gamma, beta(delta')). Reproduced when both hold, partially when only (1).

In [ ]:
key = lambda x: np.round(x, 9)
bmap = {(g, r, key(d)): a_ for g, r, d, a_ in B[["grid", "reading", "delta", "alpha"]].itertuples(index=False)}
X = Pg.copy()
X["beta_dprime"] = [bmap[(g, r, key(dp))] for g, r, dp in X[["grid", "reading", "dprime"]].itertuples(index=False)]
X["beta_delta"] = [bmap[(g, r, key(d))] for g, r, d in X[["grid", "reading", "delta"]].itertuples(index=False)]
X["pred"] = np.minimum(X.alpha, X.beta_dprime)
X["both_gt2"] = (X.alpha > 2) & (X.beta_dprime > 2)
X["eq8_10pct"] = (X.beta_delta - X.pred).abs() <= 0.1 * X.pred
ea = X.groupby(["grid", "reading"]).agg(n_pairs=("alpha", "size"), both_gt2=("both_gt2", "sum"), eq8_within10=("eq8_10pct", "sum"),
                                         median_rel_err=("pred", lambda s: np.nan)).reset_index()
ea["median_rel_err"] = [np.median(np.abs(d.beta_delta - d.pred) / d.pred) for _, d in X.groupby(["grid", "reading"])]
ea["cond1"] = ea.both_gt2 > ea.n_pairs / 2; ea["cond2"] = ea.eq8_within10 > ea.n_pairs / 2
print(ea.round(3).to_string())
print("\ngrid (a), per pair:")
print(X[X.grid == "a"][["reading", "dprime", "delta", "ratio", "alpha", "beta_dprime", "beta_delta", "pred", "both_gt2", "eq8_10pct"]].round(3).to_string())

**Result: reproduced, under the KS fits.** Both gamma and beta(delta') exceed 2 in 15 of 15 pairs
on grid (a) and 76 of 76 on grid (b), and with the exponent unbounded beta(delta) is within 10% of
min(gamma, beta(delta')) in 9 (continuous) and 12 (discrete) of 15 and in 45 and 58 of 76 pairs
(median relative error 0.058 to 0.089); capped, in 11 and 12 of 15 and 48 and 58 of 76. Under
xmin = 1 both conditions fail (both above 2 in 0 to 17 pairs).

## fig2-neuron-beta-decrease-follows-gamma

> For neuronal spikes, beta is stable below 3000 ms, where beta at the smaller timescale is below
> gamma. Above 3000 ms beta decreases, apparently driven by a decreasing gamma.

Criterion: (1) for pairs with delta < 3000 ms, beta(delta') < gamma in more than half; (2) for
delta > 3000 ms within the relevant range, beta has a negative Spearman correlation with log
delta; (3) over the same range, gamma has a negative Spearman correlation with log delta for at
least one of the three ratios. Reproduced when all three hold under one reading of O3, partially
when (1) and (2) hold. If the range above 3000 ms has fewer than three grid points, (2) and (3)
are reported and not judged. "Negative" is read as rho < 0; the p-value is shown.

In [ ]:
dec = []
for (g, r), d in X.groupby(["grid", "reading"]):
    lowp = d[d.delta < 3000]
    c1 = int((lowp.beta_dprime < lowp.alpha).sum())
    bb = Bg[(Bg.grid == g) & (Bg.reading == r) & (Bg.delta > 3000)]
    rho2, p2 = spearmanr(np.log(bb.delta), bb.alpha) if len(bb) >= 2 else (np.nan, np.nan)
    gam = {}
    for ratio_, dd in d[d.delta > 3000].groupby("ratio"):
        gam[ratio_] = (len(dd),) + (tuple(np.round(spearmanr(np.log(dd.delta), dd.alpha), 3)) if len(dd) >= 2 else (np.nan, np.nan))
    judged = len(bb) >= 3
    dec.append(dict(grid=g, reading=r, pairs_below3000=len(lowp), beta_dprime_lt_gamma=c1, cond1=bool(c1 > len(lowp) / 2),
                    n_grid_above3000=len(bb), rho_beta=round(rho2, 3), p_beta=round(p2, 4), cond2=bool(rho2 < 0) if judged else None,
                    gamma_by_ratio=gam, cond3=bool(any(v[1] < 0 for v in gam.values() if v[0] >= 2)) if judged else None))
dec = pd.DataFrame(dec); print(dec.to_string())
print("\nbeta above 3000 ms:")
print(Bg[Bg.delta > 3000].pivot_table(index=["grid", "delta"], columns="reading", values="alpha").round(3).to_string())

**Result: reproduced, on grid (b) under the KS readings of the fit; not judged on grid (a).** For
pairs with delta < 3000 ms, beta(delta') < gamma in 40 of 57 under both KS fits with the
exponent unbounded (38 for the capped continuous fit). Above 3000 ms grid (b) has 6 points; beta
falls from 2.064 to 1.867 (discrete fit), Spearman rho = -0.829 (p = 0.0416), and gamma falls
with delta for each ratio (rho -0.714, -0.964 and -0.943). Grid (a) has one point above 3000 ms,
so parts (2) and (3) are not judged there; its part (1) holds (7 of 12). Under xmin = 1, part
(1) fails (24 of 57) while parts (2) and (3) hold. Part (2) rests on 6 points at p = 0.0416.

### Variant (O2): the epochs taken separately

The claims use all spikes of the session. As a variant, the representative unit is analysed in
each epoch alone (IETs across epoch boundaries removed), under grid (b) and the KS readings.

In [ ]:
E = epochs["Achilles-10252013"]
var = []
for lab, a_, b in zip(E["label"], E["start"] * 1e3, E["stop"] * 1e3):
    s = spk[(spk >= a_) & (spk <= b)]
    rr = bt.analyse_sequence(s, "b", readings=("ks_discrete", "ks_continuous"))
    bb = pd.DataFrame(rr["bursts"]); bb = bb[bb.on_grid]
    pp = pd.DataFrame(rr["pairs"])
    for r_, d in bb.groupby("reading"):
        below = d[d.delta < 3000]
        var.append(dict(epoch=lab, reading=r_, n=rr["n"], lo=round(rr["lo"], 1), hi=round(rr["hi"], 1), n_delta=len(d),
                        b_supported=int(d.supported.sum()), median_beta_below3000=round(below.alpha.median(), 3) if len(below) else np.nan,
                        k_supported=int(pp[pp.reading == r_].supported.sum()), n_pairs=int((pp.reading == r_).sum())))
print(pd.DataFrame(var).to_string())

### Note on the variant

Taken epoch by epoch, the representative unit has 96 875 spikes in the first rest period, 44 099
in the maze and 87 382 in the second rest period, which sum to the 228 356 of Figure 2: the
paper used the whole session (O2). The maze epoch alone has a relevant range of only 55.2 to
812.3 ms and a median beta below 3000 ms of 1.776 to 1.950; the rest periods give 2.499 to 2.829.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome | Rests on |
|---|---|---|
| fig2-neuron-representative-sequence | reproduced | O1 reading (b), row 9 counted from 0 (unit id 18) |
| fig2-iet-broad | reproduced | |
| fig2-neuron-burst-size-heavy-tail | reproduced | KS fits, exponent unbounded or capped; xmin = 1 gives partial except the continuous fit on grid (a) |
| fig2-neuron-beta-stable | not reproduced | median 2.354, but beta 3.349 at 55 ms; no reading holds |
| fig2-neuron-beta-decrease-follows-gamma | reproduced | grid (b), KS fits |
| fig2-neuron-merging-heavy-tail | reproduced | discrete KS fit (interquartile range 0.378 and 0.415) |
| fig2-neuron-merging-collapse | not reproduced | assumption: exponent unbounded (max deviation 0.538); capped at 3 it passes at 0.490 |
| fig2-exponents-above-two | reproduced | KS fits |
| fig2-other-neurons-similar | reproduced | continuous fit from xmin = 1 alone (0.883 to 0.906); KS fits 0.233 to 0.454 over the pyramidal units of all sessions, 0.083 to 0.342 in Achilles-10252013 (`other_neurons.ipynb`) |
| fig2-wikipedia-beta | blocked (too_large) | |
| fig2-earthquake-beta | blocked (other) | |
| fig4-model-heavy-tails | partially reproduced | one reading for all three series; seed sensitive (`model.ipynb`) |
| fig4-model-iet-shapes | partially reproduced | seed sensitive (`model.ipynb`) |
| model-decremental-equivalent | not reproduced | literal set-up of Section IV (`model.ipynb`) |

How the analysis differs from the paper's:

- The fit is the `powerlaw` package, as the paper cites, under the four readings of O3; a claim
  holds when it holds under one of them, as O3 says. The package's release 2.0.0, locked here,
  caps the exponent at 3 by default and lets the cap move xmin; the method the paper cites
  (Clauset et al., ref. 36) has no such cap, and the paper gives no version. The fits here leave
  the exponent unbounded (an assumption), and the capped fits are shown beside them. The cap
  decides one outcome, fig2-neuron-merging-collapse.
- The grid of delta is not given in the paper. Grid (b) fits a whole number of log steps (40 of
  0.0514 for the representative unit) between the ends of the relevant range, and drops the upper
  end, which has fewer than 100 bursts, so it has 40 points from 55.2 to 5581.5 ms; every 5th point
  then gives the delta' labels of Figure 2 and every 8th the delta labels. Grid (a) has ratio 1.82
  (O4).
- The relevant range is computed exactly from the IETs (55.25 to 6282.75 ms here); Figure 2 shows
  6300 ms, which has 99 bursts by this count, below the 100 the Methods ask for (printed with the grids).

What was found about the dataset:

- The paper's "unit 9" is row 9 of the units table counted from 0 (id 18). It is labelled
  inhibitory, though the paper describes the data as pyramidal cells; under the KS fits the
  inhibitory units show the paper's pattern far more often than the pyramidal ones (`other_neurons.ipynb`). Later versions renumber
  the units, so "unit 9" cannot be found by id in any version.
- In three sessions (Buddy-06272013, Cicero-09012014, Gatsby-08022013) the spike train of unit id
  2 is not sorted: it is two sorted runs, the first of exactly 65 536 or 131 072 spikes starting
  in mid-session, the second starting at the beginning (`other_neurons.ipynb` and the data check).
- In Gatsby-08022013, 4.7% of the spikes fall after the end of the last epoch (30 413.6 s).
- The position series stores its sampling interval (0.0256 s) in the attribute `rate`, which
  NWB defines as a rate in Hz.
- The epochs are longer than the paper's description of the maze session (45 minutes): the maze
  epochs last 0.57 to 1.54 h, and the first rest periods 2.98 to 5.37 h.
- No spike train is deposited twice. The spikes and the maze position share a clock: the
  excitatory population rate correlates with running speed at r = 0.618 at zero shift, and at a
  median |r| of 0.043 beyond 60 s.

In [ ]:
print(f"notebook time {(time.time() - T0) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")